# ScienceQA RAG Pipeline Evaluation & Visualization

This notebook evaluates the ScienceQA Retrieval-Augmented Generation (RAG) pipeline on the test set, reports accuracy by grade and subject, and visualizes the results.

In [10]:
# 1. Import Required Libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Optional: for interactive plots
import plotly.express as px

## 2. Load Model and Data
Load the CustomBERTEmbedding model and the ScienceQA training and test datasets.

In [ ]:
import sys
import os

# Add the models directory to sys.path (robust for Jupyter)
models_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "models"))
if models_dir not in sys.path:
    sys.path.insert(0, models_dir)

from custom_bert import CustomBERTEmbedding

VOCAB_SIZE = 30522
EMBED_DIM = 256
NUM_HEADS = 8
NUM_LAYERS = 6
MAX_SEQ_LENGTH = 128

# Load model
model = CustomBERTEmbedding(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, NUM_LAYERS, MAX_SEQ_LENGTH)
model.load_state_dict(torch.load(os.path.join(models_dir, "custom_bert.pth"), map_location=torch.device('cpu')))
model.eval()

# Load data
data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
train_path = os.path.join(data_dir, "ScienceQA_train.json")
test_path = os.path.join(data_dir, "ScienceQA_test.json")
import json
with open(train_path, 'r', encoding='utf-8') as f:
    data_train = pd.read_json(f)
with open(test_path, 'r', encoding='utf-8') as f:
    data_test = pd.read_json(f)

print(f"Train samples: {len(data_train)} | Test samples: {len(data_test)}")

/Users/jessicalim/miniconda3/envs/ml_project/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
/var/folders/7n/sdnmyc4j33q0y4lz8rx2m3wc0000gn/T/ipykernel_48790/1342200882.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowl

Train samples: 12726 | Test samples: 4241


## 3. Embed Training Lectures
Embed the training lectures using the model if embeddings do not already exist.

In [12]:
def simple_tokenizer(text, vocab_size=VOCAB_SIZE, max_seq_length=MAX_SEQ_LENGTH):
    tokens = text.lower().split()
    ids = [abs(hash(token)) % vocab_size for token in tokens]
    if len(ids) < max_seq_length:
        ids += [0] * (max_seq_length - len(ids))
    else:
        ids = ids[:max_seq_length]
    return torch.tensor(ids, dtype=torch.long).unsqueeze(0)

def embed_text(model, text):
    input_ids = simple_tokenizer(text)
    with torch.no_grad():
        emb = model(input_ids)
        emb_vec = emb.mean(dim=1).squeeze().cpu().numpy()
    return emb_vec

def batch_embed_lectures(model, data, save_path):
    embs = []
    for lecture in tqdm(data['lecture'], desc="Embedding lectures"):
        embs.append(embed_text(model, lecture))
    embs = np.stack(embs)
    np.save(save_path, embs)
    print(f"Saved {len(embs)} lecture embeddings to {save_path}")

## 4. Load or Compute Lecture Embeddings
Load precomputed lecture embeddings from file, or compute and save them if not present.

In [13]:
emb_path = "../data/lecture_embeddings.npy"
if not os.path.exists(emb_path):
    batch_embed_lectures(model, data_train, emb_path)
data_train_embs = np.load(emb_path)
print(f"Loaded lecture embeddings: {data_train_embs.shape}")

Loaded lecture embeddings: (12726, 30522)


## 5. Evaluate on Test Set
Run the evaluation function on the test set to get predictions and statistics.

In [14]:
results_path = "../data/rag_test_results.csv"
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    print(f"Loaded evaluation results: {len(results_df)} rows")
else:
    raise FileNotFoundError(f"Results file not found: {results_path}")

results_df.head()

FileNotFoundError: Results file not found: ../data/rag_test_results.csv

## 6. Calculate Accuracy by Grade and Subject
Aggregate the results to compute accuracy for each grade and each subject.

In [ ]:
# Accuracy by grade
acc_by_grade = results_df.groupby('grade')['is_correct'].mean().sort_index()
# Accuracy by subject
acc_by_subject = results_df.groupby('subject')['is_correct'].mean().sort_values(ascending=False)

print("Accuracy by grade:")
print(acc_by_grade)
print("\nAccuracy by subject:")
print(acc_by_subject)

## 7. Visualize Accuracy by Grade
Create a bar plot showing accuracy for each grade using matplotlib or seaborn.

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(x=acc_by_grade.index, y=acc_by_grade.values, palette="Blues_d")
plt.title("RAG Accuracy by Grade")
plt.xlabel("Grade")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

## 8. Visualize Accuracy by Subject
Create a bar plot showing accuracy for each subject using matplotlib or seaborn.

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=acc_by_subject.index, y=acc_by_subject.values, palette="Greens_d")
plt.title("RAG Accuracy by Subject")
plt.xlabel("Subject")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.show()